In [4]:
import numpy as np
import pandas as pd
from statistics import NormalDist

from helpers.helpers_QR2LIM import *


# ============================================================
# Parameters: same first-limit parameters as QRNIID.ipynb
# ============================================================

q0 = 10

mu_plus = 1.0
mu_minus = 1.1
alpha = 0.50
beta = 0.50
lambda_floor = 1e-8
gamma_cross = 1.0

# Second-limit parameters requested
q2_0 = 10
lambda_second_add = 1.500
lambda_second_remove = 0.500

# Naive Monte Carlo setup
n_samples = 100_000
h_grid = [5, 10]

rng = np.random.default_rng(12345)


# ============================================================
# Naive Monte Carlo simulation
# ============================================================

samples = sample_second_limit_conditionals_constant(
    n_samples=n_samples,
    q1_0=q0,
    q2_0=q2_0,
    mu_plus=mu_plus,
    mu_minus=mu_minus,
    alpha=alpha,
    beta=beta,
    lambda_second_add=lambda_second_add,
    lambda_second_remove=lambda_second_remove,
    gamma_cross=gamma_cross,
    rng=rng,
    t_max=10_000.0,
)

plus2_at_tau = samples["plus2_at_tau"]
first_empty = samples["first_empty"]
hit_zero = samples["hit_zero"]


# ============================================================
# Conditioning on tau = tau_{+1}
# ============================================================

condition_mask = hit_zero & (first_empty == "plus1")

plus2_cond = plus2_at_tau[condition_mask]

n_cond = len(plus2_cond)

print(f"Number of simulations = {n_samples}")
print(f"Number of samples with tau = tau_{{+1}} = {n_cond}")
print(f"Empirical P(tau = tau_{{+1}}) = {n_cond / n_samples:.6f}")
print()


# ============================================================
# Estimate P(N^{+2}_tau <= h | tau = tau_{+1})
# for h in {5, 10}
# ============================================================

def binomial_confidence_interval(p_hat, n, level=0.95):
    z = NormalDist().inv_cdf(0.5 + level / 2.0)
    se = np.sqrt(p_hat * (1.0 - p_hat) / n)
    return p_hat - z * se, p_hat + z * se, se


rows = []

for h in h_grid:
    indicators = plus2_cond <= h
    count = indicators.sum()
    p_hat = indicators.mean()

    ci_low, ci_high, se = binomial_confidence_interval(p_hat, n_cond)

    rows.append({
        "h": h,
        "count_event": count,
        "n_conditioned": n_cond,
        "p_hat": p_hat,
        "std_error": se,
        "ci_95_low": ci_low,
        "ci_95_high": ci_high,
    })

results = pd.DataFrame(rows)

print(results.to_string(index=False))


# ============================================================
# Optional: useful diagnostics
# ============================================================

print()
print("Summary of N^{+2}_tau | {tau = tau_{+1}}")
print(f"Mean   = {plus2_cond.mean():.4f}")
print(f"Std    = {plus2_cond.std(ddof=1):.4f}")
print(f"Min    = {plus2_cond.min():.0f}")
print(f"Q01    = {np.quantile(plus2_cond, 0.01):.0f}")
print(f"Q05    = {np.quantile(plus2_cond, 0.05):.0f}")
print(f"Median = {np.quantile(plus2_cond, 0.50):.0f}")
print(f"Q95    = {np.quantile(plus2_cond, 0.95):.0f}")
print(f"Max    = {plus2_cond.max():.0f}")

Number of simulations = 100000
Number of samples with tau = tau_{+1} = 49998
Empirical P(tau = tau_{+1}) = 0.499980

 h  count_event  n_conditioned   p_hat  std_error  ci_95_low  ci_95_high
 5            0          49998 0.00000   0.000000   0.000000    0.000000
10           33          49998 0.00066   0.000115   0.000435    0.000885

Summary of N^{+2}_tau | {tau = tau_{+1}}
Mean   = 70.7493
Std    = 47.4242
Min    = 6
Q01    = 17
Q05    = 23
Median = 58
Q95    = 161
Max    = 622
